In [1]:
import os
import cv2
import pandas as pd

CSV_PATH = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/Csv/G.csv"
IMAGES_DIR = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/FramesFromCsv"
OUTPUT_DIR = "/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/PreviewFromCsv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_COLORS = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
    "truck": (0, 0, 255),
    "trunk": (0, 0, 255)
}

df = pd.read_csv(CSV_PATH)
df["name"] = df["name"].astype(str).str.strip().str.lower()
df["name"] = df["name"].replace({"trunk": "truck"})

image_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

for image_name in image_files:
    frame_num = int(os.path.splitext(image_name)[0].split("_")[-1])
    image_path = os.path.join(IMAGES_DIR, image_name)
    output_path = os.path.join(OUTPUT_DIR, image_name)

    img = cv2.imread(image_path)
    if img is None:
        continue

    h, w = img.shape[:2]
    rows = df[df["frame_num"] == frame_num]

    for _, row in rows.iterrows():
        class_name = str(row["name"]).strip().lower()
        color = CLASS_COLORS.get(class_name, (0, 255, 255))

        xmin = int(max(0, min(float(row["xmin"]), w - 1)))
        ymin = int(max(0, min(float(row["ymin"]), h - 1)))
        xmax = int(max(0, min(float(row["xmax"]), w - 1)))
        ymax = int(max(0, min(float(row["ymax"]), h - 1)))

        if xmax <= xmin or ymax <= ymin:
            continue

        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 2)
        cv2.putText(
            img,
            class_name,
            (xmin, max(20, ymin - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2,
            cv2.LINE_AA
        )

    cv2.imwrite(output_path, img)

print("Done")
print(OUTPUT_DIR)

Done
/Users/artemsaltanovskyi/Documents/2025-2026-2/2MCTE2-Advancded_AI/exam/AIProject2026/Data/PreviewFromCsv
